# ROI select & 80x80 Cut out

In [ ]:
import cv2
import numpy as np
import os
from ultralytics import YOLO

# --- การตั้งค่า ---
VIDEO_PATHS = [r"../data/video_real_data/test.mp4"]
# MODEL_NAME = "exp_v2_remake_e150_b8"
# MODEL_NAME = "exp_real_e20_b8"
MODEL_PATH = f"../MODEL_train/runs/detect/koi-detect/{MODEL_NAME}/weights/best.pt"
GAMMA_VALUE = 0.5    
PROCESS_EVERY_N = 2

# --- เกณฑ์การกรองใหม่ ---
MAX_WIDTH = 80
MAX_HEIGHT = 80

def adjust_gamma(image, gamma=1.0):
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255
                      for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(image, table)

# โหลด Model
model = YOLO(MODEL_PATH)

for path in VIDEO_PATHS:
    if not os.path.exists(path):
        print(f"❌ ไม่พบไฟล์: {path}")
        continue

    cap = cv2.VideoCapture(path)
    ret, first_frame = cap.read()
    if not ret: continue

    # --- ส่วนเลือก ROI ---
    first_frame_adj = adjust_gamma(first_frame, gamma=GAMMA_VALUE)
    print(f"🖱️ ลากเมาส์เลือก ROI (พื้นที่ในบ่อ) แล้วกด ENTER")
    roi = cv2.selectROI("Select ROI", first_frame_adj, fromCenter=False)
    cv2.destroyWindow("Select ROI")

    # ถ้าไม่ได้เลือกพื้นที่ ให้ใช้ทั้งภาพ
    if roi == (0, 0, 0, 0):
        roi = (0, 0, int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))

    rx, ry, rw, rh = roi
    frame_count = 0
    display_boxes = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        frame_count += 1
        processed_frame = adjust_gamma(frame, gamma=GAMMA_VALUE)
        
        # ตัดภาพเฉพาะพื้นที่ ROI
        roi_img = processed_frame[ry:ry+rh, rx:rx+rw]

        if frame_count % PROCESS_EVERY_N == 0:
            # ประมวลผล AI ใน ROI
            results = model.predict(
                roi_img, 
                conf=0.2, 
                verbose=False, 
                device=0, 
                half=True
            )
            
            display_boxes = []
            if len(results[0].boxes) > 0:
                for box in results[0].boxes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    conf = float(box.conf[0])
                    
                    # คำนวณขนาดของกล่อง
                    bw = x2 - x1
                    bh = y2 - y1

                    # --- เงื่อนไข: ถ้าใหญ่กว่า 80x80 ให้ตัดออก ---
                    if bw > MAX_WIDTH or bh > MAX_HEIGHT:
                        continue  # ข้ามกล่องนี้ไป ไม่เก็บไว้แสดงผล
                    
                    # ถ้าผ่านเงื่อนไข (ขนาดเล็กกว่า 80x80) ให้เก็บไว้
                    display_boxes.append(((x1, y1, x2, y2), conf))

        # --- ส่วนการวาดภาพ ---
        # วาดเส้นขอบเขต ROI (สีฟ้า)
        cv2.rectangle(processed_frame, (rx, ry), (rx + rw, ry + rh), (255, 0, 0), 1)

        for (x1, y1, x2, y2), conf in display_boxes:
            # ปรับพิกัดคืนค่ากลับไปที่เฟรมหลัก (Full Frame)
            cv2.rectangle(processed_frame, (x1+rx, y1+ry), (x2+rx, y2+ry), (0, 255, 0), 1)
            cv2.putText(processed_frame, f"koi {conf:.2f}", (x1+rx, y1+ry - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
        
        # แสดงจำนวนที่ผ่านการกรองแล้ว
        cv2.putText(processed_frame, f"Detected (Filtered): {len(display_boxes)}", 
                    (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        cv2.imshow("Manual ROI & Size Filter", processed_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == 27: break
        if key == ord('q'):
            cap.release()
            cv2.destroyAllWindows()
            exit()

    cap.release()

cv2.destroyAllWindows()
print("✅ ประมวลผลเสร็จสิ้น")

🖱️ ลากเมาส์เลือก ROI (พื้นที่ในบ่อ) แล้วกด ENTER
✅ ประมวลผลเสร็จสิ้น
